# Experiment 2 — Muon vs AdamW for **QLoRA** fine-tuning

Part 2 of the optimizer study. We fine-tune frozen, 4-bit-quantized 8-12B models with **LoRA**
and change only one thing: **the optimizer driving the adapters** — `torch.optim.AdamW` vs the
reference **Muon**.

**The tension that makes this interesting:** Muon orthogonalizes the gradient so the update is
*full-rank* (every singular value -> 1). LoRA *forces* the update low-rank (`ΔW = B·A`, rank r).
Muon can equalize the singular values *within* A and B, but the product BA is still rank-capped.
So does Muon still help under LoRA, or does its advantage vanish? That's the experiment.

### Before you run — read this
- **8-12B models => QLoRA is required.** We load the base in 4-bit (bitsandbytes NF4). 8-9B fits a
  16GB T4; **Gemma-4-12B is tight on 16GB — use a 24GB GPU** (Colab L4/A100) or it may OOM.
- **Confirm the repo ids** in `MODELS` (editable). Wrong id -> that model is skipped.
- **Gemma is gated** (`huggingface-cli login` + accept license). All three use `trust_remote_code`.
- **Qwen-3.5-9B and Gemma-4-12B are multimodal (VLMs).** We load them via the image-text class but
  **train text-only** — LoRA is restricted to the **language-model** linear layers (vision tower,
  projector and LM head are excluded automatically), and no images are fed. LFM2.5 is a text LM.
- **bitsandbytes is CUDA-only** — this notebook can't run on CPU.
- Different tokenizers => loss is comparable Adam-vs-Muon **within** a model, never across models.

In [ ]:
# --- Setup -------------------------------------------------------------------
import importlib, subprocess, sys
for pkg in ("transformers", "peft", "bitsandbytes", "datasets", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", pkg], check=True)

import math, time, random
import torch, torch.nn as nn

# Backport nn.Module.set_submodule for torch < 2.5 (PEFT's LoRA injection needs it).
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        *parent, last = target.split(".")
        mod = self.get_submodule(".".join(parent)) if parent else self
        setattr(mod, last, module)
    nn.Module.set_submodule = _set_submodule

assert torch.cuda.is_available(), "QLoRA needs a CUDA GPU (bitsandbytes is CUDA-only)."
device = "cuda"
print("torch", torch.__version__, "| gpu:", torch.cuda.get_device_name(0),
      "| bf16:", torch.cuda.is_bf16_supported())

# Gated models (Gemma) need a login. Prefer `huggingface-cli login` or an env var;
# do NOT hardcode a token in the notebook.
# from huggingface_hub import login; login()

In [ ]:
# --- Config (EDIT repo ids / batch sizes here) -------------------------------
# kind: "causal" = text LM; "multimodal" = VLM (load via image-text class, LoRA on the
# language-model layers only, train text-only — no images).
MODELS = {
    "lfm2.5-8b-a1b": dict(repo="LiquidAI/LFM2.5-8B-A1B",   trust_remote_code=True,  kind="causal",     micro_batch=4),
    "qwen3.5-9b":    dict(repo="Qwen/Qwen3.5-9B",          trust_remote_code=True,  kind="multimodal", micro_batch=4),
    "gemma4-12b":    dict(repo="google/gemma-4-12B-it",    trust_remote_code=True,  kind="multimodal", micro_batch=2),
}

ARMS = ["adamw", "muon"]   # add "muon-aspect" to show the LoRA up-proj over-scaling pathology

CFG = dict(
    # LoRA
    r = 16, alpha = 32, dropout = 0.05,
    # optimization
    steps = 500, grad_accum = 4, seq_len = 512,    # effective batch = grad_accum * micro_batch
    grad_checkpoint = False,                       # False = faster (you have VRAM); not related to QLoRA
    adam_lr = 2e-4, muon_lr = 2e-3, weight_decay = 0.0,
    warmup = 50, min_lr_frac = 0.1, grad_clip = 1.0,
    # data / eval / logging
    n_train = 5000, n_val = 500,
    eval_every = 50, eval_iters = 20, log_every = 25, seed = 1337,
)
CFG

## Reference Muon (single-device, `scale_mode="none"` for LoRA)

Same Muon as Part 1, with one knob: the aspect-ratio step scaling. For a normal square-ish weight
matrix it's `max(1, rows/cols)**0.5`. For a thin LoRA factor (e.g. `out x r` = 4096 x 16) that
factor is huge (~16x) and over-steps the up-projection, so for LoRA we use **`scale_mode="none"`**
(scale = 1). The `"muon-aspect"` arm exists only to *demonstrate* that pathology.

In [ ]:
# --- Reference Muon ----------------------------------------------------------
def zeropower_via_newtonschulz5(G, steps=5):
    assert G.ndim == 2
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if transposed:
        X = X.T
    return X

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=2e-3, momentum=0.95, weight_decay=0.0,
                 ns_steps=5, scale_mode="none"):
        assert scale_mode in ("none", "aspect")
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay,
                                      ns_steps=ns_steps, scale_mode=scale_mode))
    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, mom = group["lr"], group["momentum"]
            wd, ns, sm = group["weight_decay"], group["ns_steps"], group["scale_mode"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                st = self.state[p]
                if "m" not in st:
                    st["m"] = torch.zeros_like(g)
                buf = st["m"]
                buf.mul_(mom).add_(g)
                g = g.add(buf, alpha=mom)
                u = zeropower_via_newtonschulz5(g, ns)
                if wd:
                    p.mul_(1 - lr * wd)
                scale = (max(1.0, p.size(0) / p.size(1)) ** 0.5) if sm == "aspect" else 1.0
                p.add_(u.to(p.dtype), alpha=-lr * scale)

print("Muon ready. NS self-check:")
_sv = torch.linalg.svdvals(zeropower_via_newtonschulz5(torch.randn(256, 64), 5))
print(f"  singular values min={_sv.min():.3f} max={_sv.max():.3f} (want ~1)")

In [ ]:
# --- Data: OpenR1-Math-220k (hard competition math, compact reference soln) --
from datasets import load_dataset

def format_prompt(ex):
    return f"Problem:\n{ex['problem']}\n\nSolution:\n"

def tokenize_example(ex, tok, max_len):
    # Flat (problem, solution); loss only on the reference solution. Hard enough that
    # even strong models start with real loss (unlike easy grade-school math).
    sol = ex.get("solution") or ""
    if not sol.strip():
        return [], []
    eos = tok.eos_token_id
    p_ids = tok(format_prompt(ex))["input_ids"]
    o_ids = tok(sol, add_special_tokens=False)["input_ids"]
    if eos is not None:
        o_ids = o_ids + [eos]
    input_ids = (p_ids + o_ids)[:max_len]
    labels = ([-100] * len(p_ids) + o_ids)[:max_len]
    return input_ids, labels

def build_data(tok):
    raw = load_dataset("open-r1/OpenR1-Math-220k", "default",
                       split="train").shuffle(seed=CFG["seed"])
    val_raw = raw.select(range(CFG["n_val"]))
    tr_raw = raw.select(range(CFG["n_val"], CFG["n_val"] + CFG["n_train"]))
    def tok_list(split):
        out = []
        for ex in split:
            ids, lab = tokenize_example(ex, tok, CFG["seq_len"])
            if sum(1 for t in lab if t != -100) >= 1:   # keep rows with >=1 response token
                out.append((ids, lab))
        return out
    tr, va = tok_list(tr_raw), tok_list(val_raw)
    print(f"  tokenized: train={len(tr):,}  val={len(va):,}")
    assert len(tr) >= 1 and len(va) >= 1, (
        f"Almost everything filtered (train={len(tr)}, val={len(va)}). "
        f"Raise CFG['seq_len'].")
    return tr, va

def collate(batch, pad_id):
    maxlen = max(len(ids) for ids, _ in batch)
    X, L, A = [], [], []
    for ids, lab in batch:
        pad = maxlen - len(ids)
        X.append(ids + [pad_id] * pad)
        L.append(lab + [-100] * pad)
        A.append([1] * len(ids) + [0] * pad)
    return (torch.tensor(X), torch.tensor(L), torch.tensor(A))

def get_batch(data, bs, pad_id):
    assert len(data) > 0, "empty dataset after tokenization — check chat_template / seq_len"
    idx = [random.randrange(len(data)) for _ in range(bs)]   # with replacement: robust to small sets
    return collate([data[i] for i in idx], pad_id)

In [ ]:
# --- QLoRA model + optimizer wiring ------------------------------------------
import transformers as _tf
from transformers import AutoTokenizer, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Substrings that mark a module as part of the vision tower / multimodal projector,
# which we EXCLUDE from LoRA (we train text-only).
VISION_HINTS = ("vision", "visual", "image", "patch", "vit", "mm_proj",
                "multi_modal", "vision_tower", "connector", "projector", "audio")

def load_tokenizer(spec):
    trc = spec["trust_remote_code"]
    try:
        tok = AutoTokenizer.from_pretrained(spec["repo"], trust_remote_code=trc)
    except Exception:
        tok = AutoProcessor.from_pretrained(spec["repo"], trust_remote_code=trc).tokenizer
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token
    return tok

def load_base(spec):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    kw = dict(quantization_config=bnb, device_map="auto",
              trust_remote_code=spec["trust_remote_code"], torch_dtype=torch.bfloat16)
    classes = []
    if spec.get("kind") == "multimodal":   # try VLM classes first, fall back to causal
        for cn in ("AutoModelForImageTextToText", "AutoModelForMultimodalLM",
                   "AutoModelForVision2Seq"):
            if hasattr(_tf, cn):
                classes.append(getattr(_tf, cn))
    classes.append(_tf.AutoModelForCausalLM)
    err = None
    for cls in classes:
        try:
            return cls.from_pretrained(spec["repo"], **kw)
        except Exception as e:
            err = e
    raise err

def lm_linear_targets(model):
    # Full names of all Linear (incl. bnb 4-bit) layers EXCEPT vision/projector/lm_head.
    # Passing full names to PEFT targets exactly those modules (text decoder only).
    names = []
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear):
            low = name.lower()
            if name.endswith("lm_head") or any(h in low for h in VISION_HINTS):
                continue
            names.append(name)
    return names

def load_qlora(spec):
    base = load_base(spec)
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=CFG["grad_checkpoint"])
    targets = lm_linear_targets(base)
    print(f"    LoRA on {len(targets)} language-model linear layers (vision/head excluded)")
    torch.manual_seed(CFG["seed"])   # identical adapter init across arms
    lcfg = LoraConfig(r=CFG["r"], lora_alpha=CFG["alpha"], lora_dropout=CFG["dropout"],
                      target_modules=targets, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(base, lcfg)
    model.print_trainable_parameters()
    return model

def build_optims(model, arm):
    trainable = [p for p in model.parameters() if p.requires_grad]
    if arm == "adamw":
        opts = [torch.optim.AdamW(trainable, lr=CFG["adam_lr"], betas=(0.9, 0.95),
                                  weight_decay=CFG["weight_decay"])]
    else:
        scale = "aspect" if arm == "muon-aspect" else "none"
        hidden = [p for p in trainable if p.ndim == 2]    # LoRA A/B are all 2D
        rest = [p for p in trainable if p.ndim != 2]
        opts = [Muon(hidden, lr=CFG["muon_lr"], momentum=0.95,
                     weight_decay=CFG["weight_decay"], scale_mode=scale)]
        if rest:
            opts.append(torch.optim.AdamW(rest, lr=CFG["adam_lr"], betas=(0.9, 0.95)))
    for o in opts:
        for g in o.param_groups:
            g["base_lr"] = g["lr"]
    return opts

def lr_frac(step):
    if step < CFG["warmup"]:
        return step / max(1, CFG["warmup"])
    if step >= CFG["steps"]:
        return CFG["min_lr_frac"]
    prog = (step - CFG["warmup"]) / max(1, CFG["steps"] - CFG["warmup"])
    cos = 0.5 * (1 + math.cos(math.pi * prog))
    return CFG["min_lr_frac"] + (1 - CFG["min_lr_frac"]) * cos

@torch.no_grad()
def estimate_loss(model, data, bs, pad_id):
    # Deterministic: a FIXED slice of val (same examples every eval, identical across arms)
    # so baselines match and the Adam-vs-Muon gap isn't drowned in sampling noise.
    model.eval()
    tot = n = 0
    limit = min(len(data), CFG["eval_iters"] * bs)
    for i in range(0, limit, bs):
        x, lab, attn = (t.to(device) for t in collate(data[i:i + bs], pad_id))
        tot += model(input_ids=x, attention_mask=attn, labels=lab).loss.item()
        n += 1
    model.train()
    return tot / max(n, 1)

def train_run(spec, arm, train_data, val_data, pad_id):
    model = load_qlora(spec)
    opts = build_optims(model, arm)
    print(f"    arm={arm}: {sum(len(o.param_groups[0]['params']) for o in opts)} param tensors")
    model.train()
    ga, bs = CFG["grad_accum"], spec["micro_batch"]
    vl0 = estimate_loss(model, val_data, bs, pad_id)   # step-0 baseline (gauge headroom)
    print(f"      step     0  val {vl0:.3f}  ppl {math.exp(min(vl0,20)):.1f}  (baseline)", flush=True)
    log, t0 = [], time.time()
    for step in range(1, CFG["steps"] + 1):
        frac = lr_frac(step)
        for o in opts:
            for g in o.param_groups:
                g["lr"] = g["base_lr"] * frac
        for o in opts:
            o.zero_grad(set_to_none=True)
        run = 0.0
        for _ in range(ga):
            x, lab, attn = (t.to(device) for t in get_batch(train_data, bs, pad_id))
            loss = model(input_ids=x, attention_mask=attn, labels=lab).loss
            (loss / ga).backward()
            run += loss.item() / ga
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], CFG["grad_clip"])
        for o in opts:
            o.step()
        if step % CFG["log_every"] == 0:
            log.append(dict(step=step, train_loss=run, val_loss=None, sec=time.time() - t0))
        if step % CFG["eval_every"] == 0:
            vl = estimate_loss(model, val_data, bs, pad_id)
            if log:
                log[-1]["val_loss"] = vl
            print(f"      step {step:5d}  train {run:.3f}  val {vl:.3f}  "
                  f"ppl {math.exp(min(vl,20)):.1f}  {time.time()-t0:.0f}s")
    del model
    torch.cuda.empty_cache()
    return log

In [ ]:
# --- Run all models x arms ---------------------------------------------------
results = {}
for name, spec in MODELS.items():
    print(f"\n{'='*64}\n{name}  ({spec['repo']})\n{'='*64}")
    try:
        tok = load_tokenizer(spec)
        train_data, val_data = build_data(tok)
        pad_id = tok.pad_token_id if tok.pad_token_id is not None else 0
        results[name] = {}
        for arm in ARMS:
            print(f"  --- {arm.upper()} ---", flush=True)
            try:                       # per-ARM guard: a failed arm is visible & isolated
                results[name][arm] = train_run(spec, arm, train_data, val_data, pad_id)
            except Exception as e:
                print(f"  {arm} FAILED: {type(e).__name__}: {e}", flush=True)
                torch.cuda.empty_cache()
        del train_data, val_data
    except Exception as e:
        print(f"  SKIPPED {name}: {type(e).__name__}: {e}")
print("\ndone.")

In [ ]:
# --- Plot + summary ----------------------------------------------------------
import json
import matplotlib.pyplot as plt

DISPLAY = {"lfm2.5-8b-a1b": "LFM2.5-8B-A1B (MoE)", "qwen3.5-9b": "Qwen3.5-9B",
           "gemma4-12b": "Gemma 4 12B"}
COL = {"adamw": "#d62728", "muon": "#1f77b4", "muon-aspect": "#7f7f7f"}

def best_of(log):
    vs = [(e["step"], e["val_loss"]) for e in log if e.get("val_loss") is not None]
    if not vs:
        return None
    s, v = min(vs, key=lambda t: t[1])
    return dict(step=s, val=v, ppl=math.exp(min(v, 20)))

models = [m for m in results if results.get(m)]
if models:
    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5), squeeze=False)
    for ax, name in zip(axes[0], models):
        for arm, log in results[name].items():
            xs = [e["step"] for e in log if e.get("val_loss") is not None]
            ys = [e["val_loss"] for e in log if e.get("val_loss") is not None]
            if xs:
                ax.plot(xs, ys, "-o", ms=3, color=COL.get(arm, "k"), label=arm)
                b = best_of(log)
                ax.scatter([b["step"]], [b["val"]], marker="*", s=90, zorder=5,
                           color=COL.get(arm, "k"), edgecolor="white")
        ax.set_title(DISPLAY.get(name, name), fontweight="bold")
        ax.set_xlabel("step"); ax.set_ylabel("val loss"); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.savefig("qlora_val_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

print(f"\n{'model':22}{'arm':12}{'best val':>10}{'best ppl':>10}{'wall(s)':>9}")
for name in models:
    for arm, log in results[name].items():
        b = best_of(log)
        if b:
            print(f"{DISPLAY.get(name,name):22}{arm:12}{b['val']:>10.3f}"
                  f"{b['ppl']:>10.1f}{log[-1]['sec']:>9.0f}")
with open("results_qlora.json", "w") as f:
    json.dump(results, f)
print("\nsaved: qlora_val_curves.png, results_qlora.json")

## What to look for

- **Does Muon still beat AdamW under LoRA?** If yes, orthogonalizing the adapter gradients helps
  even though `BA` is rank-capped. If it washes out, that's the "LoRA neutralizes Muon" result —
  equally postable.
- **`muon` vs `muon-aspect`** (if you enable it): the aspect arm should be unstable / worse,
  demonstrating that naive Muon over-steps the thin LoRA up-projection.
- **Per-second honesty:** Newton-Schulz runs on the small adapter matrices here, so Muon's overhead
  should be *much* smaller than in full pretraining — check the wall(s) column.
- **MoE note (LFM):** LoRA adapts attention + expert linear layers; Muon on expert matrices is its
  own sub-story.
- Still single-seed, lightly-tuned. Sweep `muon_lr` in {1e-3, 2e-3, 5e-3} before concluding.